In [1]:
import torch
print(torch.cuda.is_available())

device = 'cuda' if torch.cuda.is_available() else 'cpu'

True


In [2]:
# Automatic Differentiation with torch.autograd

# backpropagation
# parameters are adjusted according to the gradient of the loss function
# with respect to the parameters.

# torch.autograd: built-in differentiation engine

import torch

x = torch.ones(5, device = device) # input tensor
#x = torch.tensor([1., 2., 3., 4., 5.]) # input tensor
y = torch.zeros(3, device = device) # expected output
# set the value of requires_grad when creating a tensor
# or later by using x.requires_grad_(True)
w = torch.randn(5, 3, requires_grad=True, device = device) # weight
b = torch.randn(3, requires_grad = True, device = device) # bias
print(w, "\n", b)
z = torch.matmul(x,w) + b 
temp = torch.sigmoid(z)
print(temp)
loss = torch.nn.functional.binary_cross_entropy_with_logits(z, y)

print(f"Gradient function for z = {z.grad_fn}")
print(f"Gradient function for loss = {loss.grad_fn}")

tensor([[ 0.0228,  0.6374, -0.4674],
        [-0.0584,  0.7160, -1.5674],
        [ 0.4911, -1.0101,  0.0961],
        [ 0.1391,  0.8424,  0.0424],
        [-0.8511, -0.5084,  0.3883]], device='cuda:0', requires_grad=True) 
 tensor([ 0.7140, -0.8461, -0.3813], device='cuda:0', requires_grad=True)
tensor([0.6124, 0.4579, 0.1313], device='cuda:0', grad_fn=<SigmoidBackward0>)
Gradient function for z = <AddBackward0 object at 0x7f3961d88160>
Gradient function for loss = <BinaryCrossEntropyWithLogitsBackward0 object at 0x7f39631b96f0>


In [3]:
# Computing Gradients

# To compute the derivatives, we call loss.backward(),
# and then retrieve the values from w.grad and b.grad

loss.backward()
print(w.grad)
print(b.grad)

tensor([[0.2041, 0.1526, 0.0438],
        [0.2041, 0.1526, 0.0438],
        [0.2041, 0.1526, 0.0438],
        [0.2041, 0.1526, 0.0438],
        [0.2041, 0.1526, 0.0438]], device='cuda:0')
tensor([0.2041, 0.1526, 0.0438], device='cuda:0')


In [4]:
# Disabling Gradient Tracking

# all tensors with requires_grad=True are tracked by autograd
# however, we can stop autograd from tracking history on Tensors
#       for example) when we just want to do forward computations
# we can stop tracking computations by torch.no_grad() block
print("torch.no_grad() block")

z = torch.matmul(x, w) + b
print(z.requires_grad)

with torch.no_grad():
    z = torch.matmul(x,w) + b
print(z.requires_grad, "\n")


# Another way to achieve the same result is to use the detach() method on the Tensor
z = torch.matmul(x,w) + b
z_det = z.detach()
print(z_det.requires_grad)

torch.no_grad() block
True
False 

False


In [5]:
# More on Computational Graphs

# autograd keeps a record of data and all excuted operations
# in a directed acyclic graph (DAG);
# In this DAG, the leaves are the input tensors, and roots are the output tensors.
# By tracing this graph from the roots to the leaves, autograd computes the gradients using the chain rule.

# The backward pass are triggered when .backward() is called on the DAG root.
# autograd then
#       computes the gradients from each .grad_fn
#       accumulated them in the respective tensor's .grad attribute
#       using the chain rule, propagets all the way to the leaf tensors